In [1]:
import os
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS

### Plot Styles

In [2]:
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 4)})

### Load data path

In [3]:
EXTRACTED_DIR = Path("../Extracted_data")   # output of extract_resumes.py
LOG_CSV       = Path("../extraction_log.csv")

print(f"Extracted dir exists: {EXTRACTED_DIR.exists()}")

Extracted dir exists: True


### Load Extracted Resumes

In [4]:
records = []

for occ_dir in sorted(EXTRACTED_DIR.iterdir()):
    if not occ_dir.is_dir():
        continue
    for txt_file in sorted(occ_dir.glob("*.txt")):
        text = txt_file.read_text(encoding="utf-8")
        # Strip page markers added by extractor
        clean = re.sub(r"--- Page \d+ (?:\(OCR\) )?---", "", text).strip()
        records.append({
            "occupation": occ_dir.name,
            "filename":   txt_file.name,
            "filepath":   str(txt_file),
            "text":       clean,
            "char_count": len(clean),
            "word_count": len(clean.split()),
            "page_count": text.count("--- Page"),
        })

df = pd.DataFrame(records)
print(f"Total resumes loaded : {len(df)}")
print(f"Occupations          : {df['occupation'].nunique()}")
print(f"\nSample:\n{df[['occupation','filename','word_count','page_count']].head(10)}")


Total resumes loaded : 2484
Occupations          : 24

Sample:
   occupation      filename  word_count  page_count
0  ACCOUNTANT  10554236.txt        3469           5
1  ACCOUNTANT  10674770.txt        1047           2
2  ACCOUNTANT  11163645.txt         628           2
3  ACCOUNTANT  11759079.txt         849           2
4  ACCOUNTANT  12065211.txt         783           2
5  ACCOUNTANT  12202337.txt         743           2
6  ACCOUNTANT  12338274.txt         810           2
7  ACCOUNTANT  12442909.txt         656           2
8  ACCOUNTANT  12780508.txt         994           2
9  ACCOUNTANT  12802330.txt         817           2


In [5]:
# Merge with extraction log for method/status metadata
if LOG_CSV.exists():
    log_df = pd.read_csv(LOG_CSV)
    log_df["_stem"] = log_df["filename"].str.replace(r"\.pdf$", "", regex=True)
    df["_stem"]     = df["filename"].str.replace(r"\.txt$", "", regex=True)
    df = df.merge(
        log_df[["_stem", "method", "status", "error"]],
        on="_stem", how="left"
    ).drop(columns="_stem")
    print("Merged extraction log")
    print(df[["filename", "method", "status"]].head())
else:
    print("extraction_log.csv not found — skipping merge")


Merged extraction log
       filename             method status
0  10554236.txt  native_pdfplumber     ok
1  10674770.txt  native_pdfplumber     ok
2  11163645.txt  native_pdfplumber     ok
3  11759079.txt  native_pdfplumber     ok
4  12065211.txt  native_pdfplumber     ok
